In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import average_precision_score, precision_recall_curve

In [ ]:
in_dir = "../../results/03_accessbpnet/16_variant_score_prediction"
out_dir = "../../results/03_accessbpnet/17_eval_variant_score"

os.makedirs(out_dir, exist_ok=True)

In [ ]:
def compute_average_precision(df):
    """
    Compute the average precision score for a given DataFrame.
    The DataFrame should contain 'y_true' and 'y_score' columns.
    """
    
    df["true_label"] = df["variant_id"].str.split(":", expand=True).iloc[:, -1]
    df["true_label"] = df["true_label"].astype(int)
    
    ap = average_precision_score(y_score=df["abs_logfc"],
                                 y_true=df["true_label"])
    
    precision, recall, _ = precision_recall_curve(y_true=df["true_label"],
                                                  y_score=df["abs_logfc"],
                                                  drop_intermediate=True)
    
    df_pr = pd.DataFrame({"precision": precision, "recall": recall})
    
    return ap, df_pr

In [ ]:
# read the data
df_atac = pd.read_csv(f"{in_dir}/ATAC.variant_scores.tsv", sep="\t")
ap, df_pr = compute_average_precision(df_atac)

df_pr.to_csv(f"{out_dir}/ATAC.variant_scores.pr.tsv", 
             sep="\t", 
             index=False)

In [ ]:
# compute precision at a given recall
mask = df_pr['recall'] >= 0.1
if mask.sum() == 0:
    _precision = 0.0
else:
    _precision = df_pr.loc[mask, 'precision'].max() if mask.sum() > 0 else 0.0
    
print(f"Precision at 0.1 recall: {_precision}")

In [ ]:
# plot precision-recall curve and add average precision score
plt.figure(figsize=(8, 6))
plt.title(f"Precision-Recall Curve (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.plot(df_pr["recall"], df_pr["precision"], label="ATAC")
plt.legend()

In [ ]:
# read the data
df_access = pd.read_csv(f"{in_dir}/ACCESS.variant_scores.tsv", sep="\t")
ap, df_pr = compute_average_precision(df_access)

df_pr.to_csv(f"{out_dir}/ACCESS.variant_scores.pr.tsv", 
             sep="\t", 
             index=False)

In [ ]:
# compute precision at a given recall
mask = df_pr['recall'] >= 0.1
if mask.sum() == 0:
    _precision = 0.0
else:
    _precision = df_pr.loc[mask, 'precision'].max() if mask.sum() > 0 else 0.0
    
print(f"Precision at 0.1 recall: {_precision}")

In [ ]:
# plot precision-recall curve at recall = 0.1
import matplotlib.pyplot as plt

recall_target = 0.1
precision_at_recall = df_pr.loc[df_pr['recall'] >= recall_target, 
                                'precision'].iloc[0]

plt.figure(figsize=(8, 6))
plt.plot(df_pr['recall'], df_pr['precision'], label='Precision-Recall curve')
plt.scatter(recall_target, precision_at_recall, color='red', label=f'Precision at recall={recall_target}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:
# plot precision-recall curve and add average precision score
plt.figure(figsize=(8, 6))
plt.title(f"Precision-Recall Curve (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.plot(df_pr["recall"], df_pr["precision"], label="ACCESS")
plt.legend()

In [ ]:
# read caQTLs
df_caqtl = pd.read_csv("../../data/Liver_caQTL/caQTL_variants_overlappingPeaks_LD-r2-0.8_withLead.bed", 
                       sep="\t")
df_caqtl["variant_id"] = df_caqtl["proxy_ID"] + ":1"
df_caqtl = df_caqtl[["variant_id", "proxy_caQTL_beta"]]
df_caqtl.rename(columns={"proxy_caQTL_beta": "beta"}, inplace=True)

In [ ]:
df_atac = df_atac[df_atac["variant_id"].isin(df_caqtl["variant_id"])].reset_index(drop=True)

In [ ]:
df_atac = pd.merge(df_atac, df_caqtl, on="variant_id")

df_atac.to_csv(f"{out_dir}/ATAC.variant_scores.merged.tsv", 
                 sep="\t", index=False)

In [ ]:
# add R squared
pearsonR, pv = stats.pearsonr(df_atac['beta'], df_atac['logfc'])

ax = sns.jointplot(
    data=df_atac,
    x="beta",
    y="logfc",
    kind = 'scatter',
    joint_kws={'marker':'o', 's':10, 'alpha':0.1, 'linewidth':0},
    marginal_kws={'bins':20, 'element':'step', 'kde':True, 'linewidth':0},
)
ax.plot_joint(sns.regplot, color="r", scatter=False, 
              line_kws={"color": "orange", 'linestyle':'dashed'})

plt.xlim(-1.5, 1.5)
plt.ylim(-0.5, 0.5)
plt.title('pearsonR = {:.3f}'.format(pearsonR))
plt.xlabel('Observed effect size (caQTL beta)')
plt.ylabel('Predicted effect size (logfc)')
plt.tight_layout()

In [ ]:
df_access = df_access[df_access["variant_id"].isin(df_caqtl["variant_id"])].reset_index(drop=True)
df_access = pd.merge(df_access, df_caqtl, on="variant_id")

In [ ]:
df_access.to_csv(f"{out_dir}/ACCESS.variant_scores.merged.tsv", 
                 sep="\t", index=False)

In [ ]:
# add R squared
pearsonR, pv = stats.pearsonr(df_access['beta'], df_access['logfc'])

ax = sns.jointplot(
    data=df_access,
    x="beta",
    y="logfc",
    kind = 'scatter',
    joint_kws={'marker':'o', 's':10, 'alpha':0.1, 'linewidth':0},
    marginal_kws={'bins':20, 'element':'step', 'kde':True, 'linewidth':0},
)
ax.plot_joint(sns.regplot, color="r", scatter=False, 
              line_kws={"color": "orange", 'linestyle':'dashed'})

plt.xlim(-1.5, 1.5)
plt.ylim(-0.5, 0.5)
plt.title('pearsonR = {:.3f}'.format(pearsonR))
plt.xlabel('Observed effect size (caQTL beta)')
plt.ylabel('Predicted effect size (logfc)')
plt.tight_layout()